In [ ]:
import json

file_path = "./GPT_Convert_Pipeline_Result.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

print(len(data))

150


In [2]:
print(data[0].keys())

dict_keys(['index', 'chapter_name', 'FQN', 'num_iter', 'Mathlib_Version', 'Tao_Version'])


In [ ]:
codes = []

for i in range(len(data)):
    # Compile Tao_Version
    codes.append(data[i]["Tao_Version"])

    # Compile Mathlib_Version
    # codes.append(data[i]["Mathlib_Version"])

print(len(codes))

150


In [ ]:
from kimina_client import KiminaClient
from kimina_client.models import Snippet

client = KiminaClient(api_url="http://127.0.0.1:8000")

snippets = [Snippet(id=str(i), code=codes[i]) for i in range(len(codes))]

resp = client.check(snippets, timeout=1000, batch_size=10)


Batches:   0%|          | 0/15 [elapsed: 00:00, remaining: ?]

In [ ]:
# Check whether it is compilable. No error -> compilable
def can_run(repl_resp) -> bool:
    resp = repl_resp.response
    messages = getattr(resp, "messages", None) or resp.get("messages", [])
    return not any(
        (getattr(m, "severity", None) or m.get("severity")) == "error"
        for m in messages
    )

In [ ]:

num_error = 0

error_data = []

for r in resp.results:
    print("###############################################")
    print(r)
    if_run = can_run(r)
    print(if_run)

    if if_run != True:
        # print("###############################################")
        # print(r)
        error_data.append(r)
        num_error += 1

print("num_error:", num_error)
print(len(error_data))

###############################################
id='0' time=0.820227 error=None response={'env': 2, 'messages': [{'severity': 'warning', 'pos': {'line': 100, 'column': 17}, 'endPos': {'line': 100, 'column': 19}, 'data': 'unused variable `ts`\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`'}, {'severity': 'warning', 'pos': {'line': 120, 'column': 4}, 'endPos': {'line': 120, 'column': 24}, 'data': "declaration uses 'sorry'"}, {'severity': 'warning', 'pos': {'line': 136, 'column': 9}, 'endPos': {'line': 136, 'column': 32}, 'data': "declaration uses 'sorry'"}], 'sorries': [{'pos': {'line': 122, 'column': 13}, 'endPos': {'line': 122, 'column': 18}, 'goal': 'inst✝ : Chapter3.SetTheory\n⊢ Function.Injective fun p =>\n    @DFunLike.coe (@Chapter3.SetTheory.Set inst✝ ↪ @Chapter3.SetTheory.Object inst✝) (@Chapter3.SetTheory.Set inst✝)\n      (fun x => @Chapter3.SetTheory.Object inst✝) Function.instFunLikeEmbedding\n      (@Chapter3.SetTheory.set_to_object inst